# Fraise Tokenizer Visualization

Discover tokenizer JSON files in data/tokenizers automatically, then inspect one or compare several. The notebook uses metadata.json for labels when it is present, but does not assume tokenizer filenames, vocabulary sizes, or a single saved tokenizer.

In [ ]:
import json
from html import escape
from pathlib import Path

from IPython.display import HTML, display
from tokenizers import Tokenizer


def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "tokenizer"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the Fraise project root from the notebook's working directory."
    )


PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
TOKENIZER_DIR = PROJECT_ROOT / "data" / "tokenizers"
METADATA_FILE = TOKENIZER_DIR / "metadata.json"


def _read_tokenizer_metadata() -> dict[Path, dict]:
    if not METADATA_FILE.is_file():
        return {}

    with METADATA_FILE.open(encoding="utf-8") as metadata_file:
        records = json.load(metadata_file)

    if isinstance(records, dict):
        records = records.get("tokenizers", [])

    by_path = {}
    if isinstance(records, list):
        for record in records:
            if not isinstance(record, dict):
                continue
            path_value = record.get("path")
            if not isinstance(path_value, str) or not path_value:
                continue
            path = Path(path_value)
            if not path.is_absolute():
                path = TOKENIZER_DIR / path
            by_path[path.resolve()] = record
    return by_path


metadata_by_path = _read_tokenizer_metadata()
tokenizer_paths = (
    sorted(
        path for path in TOKENIZER_DIR.rglob("*.json")
        if path.resolve() != METADATA_FILE.resolve()
    )
    if TOKENIZER_DIR.is_dir()
    else []
)

TOKENIZERS = {}
load_errors = []
for path in tokenizer_paths:
    try:
        tokenizer = Tokenizer.from_file(str(path))
    except Exception as error:
        load_errors.append((path, error))
        continue

    relative_path = path.relative_to(TOKENIZER_DIR).as_posix()
    record = metadata_by_path.get(path.resolve(), {})
    details = []
    if record.get("dataset"):
        details.append(str(record["dataset"]))
    if record.get("vocab_size") is not None:
        details.append(f"requested vocab {record['vocab_size']}")
    label = relative_path
    if details:
        label += f" ({', '.join(details)})"
    TOKENIZERS[label] = tokenizer

if not TOKENIZERS:
    raise FileNotFoundError(
        f"No loadable tokenizer JSON files were found under {TOKENIZER_DIR}. "
        "Create or copy a tokenizer there, then rerun this cell."
    )

print(f"Loaded {len(TOKENIZERS)} tokenizer(s):")
for name, tokenizer in TOKENIZERS.items():
    print(f"  {name} — {tokenizer.get_vocab_size():,} tokens")

if load_errors:
    print(f"\nSkipped {len(load_errors)} JSON file(s) that were not valid tokenizer files:")
    for path, error in load_errors:
        print(f"  {path.name}: {error}")

## Visualization helpers

Token offsets show which characters in the original input belong to each token. The table also includes each tokenizer's token ID and token string.

In [ ]:
def _visible_text(text: str) -> str:
    """Make whitespace visible while preserving the text."""
    return (
        text.replace(" ", "·")
        .replace("\n", "↵\n")
        .replace("\t", "⇥")
    )


def visualize_tokenization(text: str, tokenizer: Tokenizer, name: str = "Tokenizer"):
    encoding = tokenizer.encode(text)

    # Offsets let the view display the original input exactly, including when
    # a byte-level token by itself does not decode to a complete Unicode char.
    token_texts = []
    for token_id, (start, end) in zip(encoding.ids, encoding.offsets):
        token_texts.append(
            text[start:end] if end > start else tokenizer.decode([token_id])
        )

    palette = [
        "#fde68a", "#bfdbfe", "#bbf7d0", "#fecaca",
        "#ddd6fe", "#fed7aa", "#bae6fd", "#fbcfe8",
    ]

    spans = []
    for i, (token_id, token_text) in enumerate(zip(encoding.ids, token_texts)):
        color = palette[i % len(palette)]
        label = escape(_visible_text(token_text))
        spans.append(
            f'<span title="id={token_id}" '
            f'style="display:inline-block; background:{color}; color:#111827; '
            f'padding:4px 6px; margin:2px; border-radius:5px; '
            f'font-family:monospace; white-space:pre-wrap;">{label}</span>'
        )

    rows = []
    for i, (token_id, token_repr, token_text) in enumerate(
        zip(encoding.ids, encoding.tokens, token_texts)
    ):
        rows.append(
            "<tr>"
            f"<td>{i}</td>"
            f"<td><code>{token_id}</code></td>"
            f"<td><code>{escape(token_repr)}</code></td>"
            f"<td><code>{escape(_visible_text(token_text))}</code></td>"
            f"<td>{len(token_text.encode('utf-8'))}</td>"
            "</tr>"
        )

    n_chars = len(text)
    n_bytes = len(text.encode("utf-8"))
    n_tokens = len(encoding.ids)
    html = f"""
    <div style="font-family:system-ui,sans-serif; line-height:1.5">
      <h3 style="margin-bottom:8px">{escape(name)}</h3>
      <div style="margin-bottom:12px">
        <strong>{n_tokens}</strong> tokens &nbsp;·&nbsp;
        <strong>{n_chars}</strong> characters &nbsp;·&nbsp;
        <strong>{n_bytes}</strong> UTF-8 bytes &nbsp;·&nbsp;
        <strong>{n_bytes / max(n_tokens, 1):.2f}</strong> bytes/token
      </div>
      <div style="padding:12px; border:1px solid #d1d5db; border-radius:8px; margin-bottom:16px">
        {''.join(spans)}
      </div>
      <table style="border-collapse:collapse; width:100%; font-size:14px">
        <thead>
          <tr>
            <th style="text-align:left; border-bottom:1px solid #aaa; padding:6px">#</th>
            <th style="text-align:left; border-bottom:1px solid #aaa; padding:6px">ID</th>
            <th style="text-align:left; border-bottom:1px solid #aaa; padding:6px">Tokenizer token</th>
            <th style="text-align:left; border-bottom:1px solid #aaa; padding:6px">Input text</th>
            <th style="text-align:left; border-bottom:1px solid #aaa; padding:6px">Bytes</th>
          </tr>
        </thead>
        <tbody>{''.join(rows)}</tbody>
      </table>
    </div>
    """
    display(HTML(html))
    return encoding


def compare_tokenizations(text: str, tokenizer_names):
    """Render the same input with each selected tokenizer."""
    for name in tokenizer_names:
        visualize_tokenization(text, TOKENIZERS[name], name)


def show_raw_tokens(text: str, tokenizer_names):
    """Print token IDs and tokenizer token strings for selected tokenizers."""
    for name in tokenizer_names:
        encoding = TOKENIZERS[name].encode(text)
        print(f"\n{name}")
        print("IDs:", encoding.ids)
        print("Tokenizer tokens:", encoding.tokens)
        print("Decoded:", TOKENIZERS[name].decode(encoding.ids))

## Try text and compare tokenizers

Run the next cell. It will prompt for text and for tokenizer numbers in the notebook output. Enter multiple numbers separated by commas to compare tokenizers. These answers are runtime input; you do not need to edit or save the notebook source to try new words.

In [ ]:
tokenizer_names = list(TOKENIZERS)
print("Available tokenizers:")
for number, name in enumerate(tokenizer_names, start=1):
    print(f"  {number}. {name}")

test_text = input("Text to tokenize (for example, strawberry): ")
selection = input("Tokenizer number(s), comma-separated [1]: ").strip() or "1"

try:
    numbers = list(dict.fromkeys(int(part.strip()) for part in selection.split(",")))
    if not numbers or any(number < 1 or number > len(tokenizer_names) for number in numbers):
        raise ValueError
except ValueError as error:
    raise ValueError(f"Enter tokenizer numbers from 1 to {len(tokenizer_names)}, separated by commas.") from error

selected_names = [tokenizer_names[number - 1] for number in numbers]
compare_tokenizations(test_text, selected_names)

## Raw token IDs

Use show_raw_tokens from any cell to inspect IDs and tokenizer token strings without the HTML view. For example: show_raw_tokens("naïve café", list(TOKENIZERS)).